In [21]:
import torch

torch.serialization.add_safe_globals([slice])

In [22]:
import logging
import os
import random
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from data.losses import TotalLoss
from e3nn.nn import Gate
from e3nn.o3 import FullyConnectedTensorProduct, Irrep, Irreps, SphericalHarmonics
from mace.data import (
    AtomicData,
    Configuration,
    Configurations,
    KeySpecification,
    load_from_xyz,
)
from mace.modules import (
    EquivariantProductBasisBlock,
    LinearNodeEmbeddingBlock,
    RadialEmbeddingBlock,
)
from mace.modules.blocks import RealAgnosticResidualInteractionBlock
from mace.tools import AtomicNumberTable
from mace.tools.torch_geometric.dataloader import DataLoader
from numpy.typing import NDArray
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from torch import Tensor, nn
from torch.nn import CrossEntropyLoss
from torch.nn.utils import clip_grad_norm_
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_ema import ExponentialMovingAverage
from torch_geometric.data import Data
from torch_geometric.utils import scatter
from tqdm import tqdm

from mace_classifier.data.batchloader import FamilySampler, PrototypeDataset

logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

In [23]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [24]:
class MACEClassifier(nn.Module):
    def __init__(
        self,
        r_max: float,
        num_bessel: int,
        num_polynomial_cutoff: int,
        max_ell: int,
        num_interactions: int,
        num_elements: int,
        hidden_irreps: Irreps,
        correlation: int,
        avg_num_neighbors: float,
        atomic_numbers: list[int],
        radial_MLP: list[int],
        
        # TODO: maybe nicier to make head_hidden a list
        head_hidden: int,
        head_emb_size: int,
        classify_hidden: int,
        bravais_classes: int,
    ) -> None:
        super().__init__()

        self.register_buffer(
            "atomic_numbers", torch.tensor(atomic_numbers, dtype=torch.int64)
        )

        # Initial node embeddings: only 0e, think of it as nn.Embedding
        self.num_features = hidden_irreps.count(Irrep(0, 1))
        node_attrs_irreps = Irreps([(num_elements, (0, 1))])
        node_feats_irreps_init = Irreps([(self.num_features, (0, 1))])
        
        self.node_embedding = LinearNodeEmbeddingBlock(
            irreps_in=node_attrs_irreps,
            irreps_out=node_feats_irreps_init,
        )

        # Radial embedding: bessel encoding with envelope
        self.radial_embedding = RadialEmbeddingBlock(
            r_max=r_max,
            num_bessel=num_bessel,
            num_polynomial_cutoff=num_polynomial_cutoff,
        )
        edge_feats_irreps = Irreps([(self.radial_embedding.out_dim, (0, 1))])

        # Note that interaction_irreps is the intermediate 
        sh_irreps = Irreps.spherical_harmonics(max_ell)
        interaction_irreps = (sh_irreps * self.num_features).sort()[0].simplify() # type: ignore

        self.spherical_harmonics = SphericalHarmonics(
            sh_irreps, normalize=True, normalization="component"
        )

        # We use RealAgnosticInteractionBlock as our interaction portion
        self.interactions = nn.ModuleList()
        self.products = nn.ModuleList()

        for i in range(num_interactions):
            in_irreps = hidden_irreps if i > 0 else node_feats_irreps_init
            out_irreps = hidden_irreps if i < num_interactions - 1 else Irreps([(self.num_features, (0, 1))])

            interaction = RealAgnosticResidualInteractionBlock(
                node_attrs_irreps=node_attrs_irreps,
                node_feats_irreps=in_irreps,
                edge_attrs_irreps=sh_irreps,
                edge_feats_irreps=edge_feats_irreps,
                target_irreps=interaction_irreps,
                hidden_irreps=out_irreps,
                avg_num_neighbors=avg_num_neighbors,
                radial_MLP=radial_MLP
            )
            prod = EquivariantProductBasisBlock(
                node_feats_irreps=interaction_irreps,
                target_irreps=out_irreps,
                correlation=correlation,
                num_elements=num_elements,
                use_sc=True
            )
            self.interactions.append(interaction)
            self.products.append(prod)
        
        # Single head for classification (for now)
        total_scalar_dim = self.num_features * num_interactions
        
        self.structural_head = nn.Sequential(
            nn.Linear(total_scalar_dim, head_hidden),
            nn.SiLU(),
            nn.Linear(head_hidden, head_hidden),
            nn.SiLU(),
            nn.Linear(head_hidden, head_emb_size),
        )
        # Note: no bias because ranking loss makes bias have no effect
        self.structural_disorder_proj = nn.Linear(head_emb_size, 1, bias=False)

        self.bravais_classify = nn.Sequential(
            nn.Linear(head_emb_size, classify_hidden),
            nn.SiLU(),
            nn.Linear(classify_hidden, bravais_classes),
        )
        self.chemical_head = nn.Sequential(
            nn.Linear(total_scalar_dim, head_hidden),
            nn.SiLU(),
            nn.Linear(head_hidden, head_hidden),
            nn.SiLU(),
            nn.Linear(head_hidden, head_emb_size),
        )
        self.chemical_disorder_proj = nn.Linear(head_emb_size, 1, bias=False)
        
    def forward(self, data: dict[str, Tensor]) -> dict[str, Tensor]:
        # Prepare
        sources = data["edge_index"][0]
        dests = data["edge_index"][1]

        vectors = data["positions"][dests] - data["positions"][sources] + data["shifts"]
        lengths = torch.linalg.norm(vectors, dim=-1, keepdim=True)
        node_init_attrs = data["node_attrs"]
        edge_index = data["edge_index"]

        # Embeddings
        node_feats = self.node_embedding(node_init_attrs)
        edge_attrs = self.spherical_harmonics(vectors)
        edge_feats, cutoff = self.radial_embedding(
            lengths, node_init_attrs, edge_index, self.atomic_numbers
        )

        # Interaction
        scalar_feats_list: list[Tensor] = []

        for interaction, product in zip(self.interactions, self.products):
            node_feats, sc = interaction(
                node_attrs=node_init_attrs,
                node_feats=node_feats,
                edge_attrs=edge_attrs,
                edge_feats=edge_feats,
                edge_index=edge_index,
                cutoff=cutoff,
            )
            node_feats = product(
                node_feats=node_feats,
                sc=sc,
                node_attrs=node_init_attrs
            )

            # Important: scalars should all be at the front
            scalar_feats_list.append(node_feats[:, : self.num_features])
        
        node_inv_embs = torch.cat(scalar_feats_list, dim=-1)

        z_S = self.structural_head(node_inv_embs)
        z_C = self.chemical_head(node_inv_embs)
        s_hat = self.structural_disorder_proj(z_S).squeeze(-1)
        c_hat = self.chemical_disorder_proj(z_C).squeeze(-1)
        y_bravais = self.bravais_classify(z_S)

        return {
            "z_S": z_S, 
            "z_C": z_C,
            "s_hat": s_hat,
            "c_hat": c_hat,
            "y_bravais": y_bravais,
            "d_i": node_inv_embs,
        }

In [25]:
set_seed()

manifest = {
    "mp-2258": {
        "cif_file": "mp-2258_Cu3Au.cif",
        "bravais_label": "cF",
        "ordering_type_label": "l12_cu3au",
    },
    "mp-571": {
        "cif_file": "mp-571_TiNi.cif",
        "bravais_label": "cP",
        "ordering_type_label": "b2_tini",
    },
}

dataset = PrototypeDataset(
    cif_dir="data/tests",
    manifest=manifest,
    sigma_levels=[0.05, 0.10, 0.15, 0.20, 0.25],
    shuffle_levels=[0.1, 0.2, 0.3, 0.4, 0.5],
    d_nn_range=(1.0, 5.0),
)

loader = DataLoader(
    dataset, 
    batch_sampler=FamilySampler(
        dataset,
        batch_size=32,
    )
)

Mapped species Cu->1, Au->2
Supercell replication factors: [3, 3, 3], total atoms: 108
Mapped species Ti->1, Ni->2
Supercell replication factors: [4, 4, 4], total atoms: 128


In [26]:
R_MAX = 5.0
Z_TABLE = AtomicNumberTable([1, 2])

EPOCHS = 150
LR = 5e-3
WEIGHT_DECAY = 5e-7
MAX_GRAD_NORM = 10.0

LAMBDA_INV = 1.0
LAMBDA_VAR = 5.0
LAMBDA_COV = 1.0
LAMBDA_REP_C = 5.0 
LAMBDA_RANK = 15.0
LAMBDA_CLS_S = 5.0 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get average neighbors
total_nodes = 0
total_edges = 0

for data in dataset:
    total_nodes += data.positions.shape[0]
    total_edges += data.edge_index.shape[1]

avg_neighbors = total_edges / total_nodes

In [27]:
model = MACEClassifier(
    r_max=R_MAX,
    num_bessel=8,
    num_polynomial_cutoff=5,
    max_ell=2,
    num_interactions=2,
    num_elements=2,
    hidden_irreps=Irreps("32x0e + 32x1o"),
    correlation=2,
    avg_num_neighbors=avg_neighbors,
    atomic_numbers=[1, 2],
    radial_MLP=[64, 64, 64],
    head_hidden=64,
    head_emb_size=16,
    classify_hidden=32,
    bravais_classes=14,
)

In [28]:
def make_optimizer(model: MACEClassifier, lr: float, weight_decay: float) -> Adam:
    return Adam(
        [
            {"name": "embedding", "params": model.node_embedding.parameters(), "weight_decay": 0.0},
            {"name": "interactions", "params": model.interactions.parameters(), "weight_decay": weight_decay},
            {"name": "products", "params": model.products.parameters(), "weight_decay": weight_decay},
            {"name": "structural_head", "params": model.structural_head.parameters(), "weight_decay": 0.0},
            {"name": "chemical_head", "params": model.chemical_head.parameters(), "weight_decay": 0.0},
            {"name": "s_proj", "params": model.structural_disorder_proj.parameters(), "weight_decay": 0.0},
            {"name": "c_proj", "params": model.chemical_disorder_proj.parameters(), "weight_decay": 0.0},
            {"name": "bravais_clf", "params": model.bravais_classify.parameters(), "weight_decay": 0.0},
        ],
        lr=lr,
        amsgrad=True,
        betas=(0.9, 0.999),
    )

def train(model, loader, device):
    set_seed()

    # Note that our logging will be relative to global step
    wandb.init(
        project="mace-classifier",
        name="small-data-test",
        config={
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "epochs": EPOCHS,
            "lambda_inv": LAMBDA_INV,
            "lambda_var": LAMBDA_VAR,
            "lambda_cov": LAMBDA_COV,
            "lambda_rep_C": LAMBDA_REP_C,
            "lambda_rank": LAMBDA_RANK,
            "lambda_cls_S": LAMBDA_CLS_S,
        },
    )
    wandb.define_metric("epoch")
    wandb.define_metric("epoch/*", step_metric="train/epoch")

    model = model.to(device)

    criterion = TotalLoss(
        lambda_inv=LAMBDA_INV,
        lambda_var=LAMBDA_VAR,
        lambda_cov=LAMBDA_COV,
        lambda_rep_C=LAMBDA_REP_C,
        lambda_rank=LAMBDA_RANK,
        lambda_cls_S=LAMBDA_CLS_S,
    )
    optimizer = make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

    global_step = 0

    for epoch in range(EPOCHS):
        model.train()
        loader.dataset.set_epoch(epoch)
        loader.dataset.set_batch(0)

        epoch_loss = 0.0
        epoch_loss_parts = defaultdict(float)
        batch_count = 0

        for batch_idx, batch in tqdm(enumerate(loader)):
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            loss, loss_dict = criterion(out, batch)
            loss.backward()
            
            grad_norm = clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()

            log_dict = {
                "train/loss": loss.item(),
                "train/grad_norm": float(grad_norm),
                "train/epoch": epoch,
                "train/batch_idx": batch_idx,
                "train/lr": optimizer.param_groups[0]["lr"],
            }

            epoch_loss += loss.item()
            for name, v in loss_dict.items():
                epoch_loss_parts[name] += v.item()
                log_dict[f"train/{name}"] = v.item()

            wandb.log(log_dict, step=global_step)

            global_step += 1
            batch_count += 1

            loader.dataset.set_batch(batch_idx + 1)

        epoch_log = { "epoch/loss": epoch_loss / batch_count }
        for name, total_value in epoch_loss_parts.items():
            epoch_log[f"epoch/{name}"] = total_value / batch_count

        wandb.log(epoch_log, step=global_step)

        global_step += 1
        scheduler.step(epoch_loss / batch_count)

In [29]:
train(model, loader, device)

epoch/cls_S,██▇▄▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/cov_C,▁▁▁▁█▁▁▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
epoch/cov_S,▁▁▁▄█▆▃▂▂▃▂▃▂▂▂▂▂▃▃▂▂▃▃▂▂▃▂▂▃▂▃▂▂▃▂▂▂▃▂▂
epoch/inv_C,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▄▃▆▆▆▅▆▆▇▇▆▇▅▇▆▅█▅▇▆▆█▆▅▆
epoch/inv_S,▁▁▅▇█▇▃▃▄▄▃▃▄▄▃▃▃▃▃▃▂▃▃▃▃▃▃▃▃▃▂▃▃▂▃▃▃▃▃▂
epoch/loss,▆▅█▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/rank_C,▂▅▇▅█▂▃▃▂▃▂▂▄▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/rank_S,██████▇▇▅▃▃▂▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/var_C,▁▁▂█▇▁▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/var_S,█▁▁▁▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁
+14,...


TypeError: TotalLoss.__init__() got an unexpected keyword argument 'lambda_rep_C'